# MODFLOW 6: inicializando una simulación.


## Objetivo.

El propósito de este documento es describir cómo se realiza una simulación de flujo usando el modelo GWF de MODFLOW 6. 

Esta descripción combina los conceptos descritos en la documentación de MODFLOW 6 y en el software [***flopy***](https://www.usgs.gov/software/flopy-python-package-creating-running-and-post-processing-modflow-based-models) ([repositorio](https://github.com/modflowpy/flopy), [documentación](https://flopy.readthedocs.io/en/3.3.2/index.html)). El software flopy permite simplificar la generación de los archivos de entrada para una simulación a través de Python, además de realizar la ejecución de la simulación y el post-procesamiento de la salida.

No se hace una descripción detallada, sino que solo se explican los conceptos principales y se relacionan con los archivos de entrada requeridos por MODFLOW 6 y con los objetos de flopy.

Se hace una configuración de una simulación agregando las componentes *Timing module*, *Numerical Solution* (IMS) y un *GWF model*, usando parámetros reducidos.

<p xmlns:cc="http://creativecommons.org/ns#" xmlns:dct="http://purl.org/dc/terms/"><a property="dct:title" rel="cc:attributionURL" href="https://github.com/luiggix/RTWMA/">MODFLOW 6: inicializando una simulación</a> by <b>Luis M. de la Cruz Salas (2025)</b> is licensed under <a href="http://creativecommons.org/licenses/by-sa/4.0/?ref=chooser-v1" target="_blank" rel="license noopener noreferrer" style="display:inline-block;">Attribution-ShareAlike 4.0 International<img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/cc.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/by.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/sa.svg?ref=chooser-v1"></a>.</p> 

<div class="alert alert-block alert-info">

## Ejemplo. Definición de las tres componentes de una simulación con flopy.

Para definir las tres componentes mostradas en la figura 2 de la notebook [00_MF6_Intro.ipynb](00_MF6_Intro.ipynb) hacemos uso de la biblioteca *flopy*. Realizaremos este proceso para un ejemplo simple, cuya única utilidad es explicar el proceso (sin resolver un problema de flujo, aún).

</div>

### Paso 1. Inicialización de la simulación.

La simulación se inicia creando un objeto de la clase [`flopy.mf6.MFSimulation`](https://flopy.readthedocs.io/en/3.3.2/source/flopy.mf6.modflow.mfsimulation.html) para cargar, construir y salvar los archivos de una simulación de MODFLOW 6.

Se debe crear un objeto de este tipo antes que cualquier otro objeto de las componentes o paquetes de MODFLOW 6 que se vayan a usar en la simulación.

A continuación creamos el objeto `sim` de la clase `flopy.mf6.MFSimulation` como sigue:



In [1]:
import flopy

In [2]:
sim = flopy.mf6.MFSimulation(
    sim_name = "flow", 
    exe_name = "../../bin/windows/mf6", 
    sim_ws   = "./sandbox1" 
)


* Una vez creado un objeto de esta clase, los demás objetos que contribuyen a la simulación, se deben enlazar al objeto de la simulación `sim`.

* Al ejecutar la celda de código anterior, se crea la carpeta `sandbox1` en donde se almacenarán todos los archivos, de entrada y de salida, que genera MODFLOW 6. Este es el espacio de trabajo (*workspace*) que por ahora estará vacío.

* El nombre de la simulación será `flow` y este nombre será usado para generar archivos de entrada y salida, con las extensiones correspondientes.

* Con la instrucción `print(sim)` es posible imprimir información de los atributos del objeto `sim`, como se hace en la siguiente celda:


In [3]:
print(sim)

sim_name = flow
sim_path = C:\Users\luiggi\Documents\GitSites\xmf6\src\examples\01_flujo_1D\sandbox1
exe_name = ../../bin/windows/mf6

###################
Package mfsim.nam
###################

package_name = mfsim.nam
filename = mfsim.nam
package_type = nam
model_or_simulation_package = simulation
simulation_name = flow





<div class="alert alert-block alert-info">

### Paso 2. Discretización temporal.

Es necesario definir los parámetros para gestionar el tiempo de la simulación. En este caso usaremos:

* Unidades: `DAYS`
* `NPER` $= 1$
* `(PERLEN, NSTP, TSMULT )` $ = (1.0, 1, 1.0)$

</div>

In [5]:
tdis = flopy.mf6.ModflowTdis(
    simulation = sim,
    time_units = "DAYS",
    nper = 1,
    perioddata = [(1.0, 1, 1.0)]
)

<div class="alert alert-block alert-info">

* Aquí se construye el objeto `tdis` que es de tipo [`flopy.mf6.ModflowTdis`](https://flopy.readthedocs.io/en/latest/source/flopy.mf6.modflow.mftdis.html).
* El primer parámetro es el objeto `sim`, es decir la simulación. De esta manera la simulación conoce los parámetros de la discretización del tiempo.
* El parámetro `perioddata` es una lista que contiene tuplas, cada una de ellas con los datos `(PERLEN, NSTP, TSMULT)` para cada periodo de estrés.
* También es posible usar la instrucción `print(sim)` para imprimir información de los atributos de este objeto, veamos:
</div>

In [6]:
print(tdis)

package_name = flow.tdis
filename = flow.tdis
package_type = tdis
model_or_simulation_package = simulation
simulation_name = flow

Block options
--------------------
time_units
{internal}
(days)


Block dimensions
--------------------
nper
{internal}
(1)


Block perioddata
--------------------
perioddata
{internal}
([(1., 1, 1.)])





<div class="alert alert-block alert-info">

* Obsérva que se usa el nombre `flow.tdis` para almacenar la información del tiempo, esto es porque el nombre de la simulación es `flow`.
* Para generar este archivo hacemos lo siguiente (recuerda que este archivo se almacena en la carpeta del espacio de trabajo, en este caso en `sandbox1`):
</div>

In [7]:
sim.write_simulation()

writing simulation...
  writing simulation name file...
  writing simulation tdis package...


<div class="alert alert-block alert-info">

* La instrucción anterior genera el archivo `mfsim.nam` que es necesario para ejecutar la simulación. Este archivo contendrá información de los objetos (paquetes) que contribuyen a la simulación. El contenido actual de este archivo debe ser como sigue:

```
# File generated by Flopy version 3.9.2 on 04/20/2025 at 12:02:33.
BEGIN options
END options

BEGIN timing
  TDIS6  flow.tdis
END timing

BEGIN exchanges
END exchanges
```
* Nota que solo se ha incluido la información del módulo de tiempo. Esta información será actualizada más adelante.
* Se genera también el archivo `flow.tdis` con la información para la discretización temporal. El contenido de este archivo es el siguiente:

```
# File generated by Flopy version 3.9.2 on 04/20/2025 at 12:02:33.
BEGIN options
  TIME_UNITS  days
END options

BEGIN dimensions
  NPER  1
END dimensions

BEGIN perioddata
       1.00000000  1       1.00000000
END perioddata
```

</div>

<div class="alert alert-block alert-info">

### Paso 3. Solución numérica.

Para cada modelo se requiere un objeto que calcule la solución numérica. 

En la celda siguiente se define un objeto de la clase [`flopy.mf6.ModflowIms`](https://flopy.readthedocs.io/en/3.3.2/source/flopy.mf6.modflow.mfims.html), se imprime la información y se escriben los archivos correspondientes.
</div>

In [8]:
ims = flopy.mf6.ModflowIms(simulation = sim)

print(ims)
sim.write_simulation()

package_name = ims_-1
filename = flow.ims
package_type = ims
model_or_simulation_package = simulation
simulation_name = flow


writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_-1...


<div class="alert alert-block alert-info">

* El objeto `ims` representa a la solución numérica; el único parámetro que se usó en este ejemplo fue el objeto de la simulación para ligar la solución numérica con el modelo que se va a resolver.
* Se pueden agregar muchos más parámetros para configurar los métodos iterativos de solución.
* En este caso la información se guarda en el archivo `flow.ims` y su contenido es mínimo debido a que no usan más parámetros:

```
# File generated by Flopy version 3.9.2 on 04/20/2025 at 12:18:32.
BEGIN options
END options
```
</div>

<div class="alert alert-block alert-info">

### Paso 4. Modelo GWF.

Ahora vamos a agregar un modelo numérico a la simulación. Para ello creamos un objeto de la clase [`flopy.mf6.ModflowGwf`](https://flopy.readthedocs.io/en/3.3.2/source/flopy.mf6.modflow.mfgwf.html) para agregar un modelo GWF como sigue:

</div>

In [9]:
gwf = flopy.mf6.ModflowGwf(
    simulation = sim,
    modelname = "flow",
    model_nam_file = "flow.nam",
)

print(gwf)
sim.write_simulation()

name = flow
model_type = gwf6
version = mf6
model_relative_path = .


writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_-1...
  writing model flow...
    writing model name file...


<div class="alert alert-block alert-info">

* Observa que el primer parámetro también es el objeto de la simulación `sim`.
* Adicionalmente se agrega el nombre del modelo, que en este caso es igual al de la simulación, y el nombre del archivo donde se guardará la información del modelo GWF.
* El código de la celda anterior crea el archivo `flow.nam` que por ahora tiene un contenido muy simple:

```
# File generated by Flopy version 3.9.2 on 04/20/2025 at 12:34:26.
BEGIN options
END options
```
* La instrucción `sim.write_simulation()` actualiza también el archivo `mfsim.nam` con la información del solucionador y del modelo GWF:

```
# File generated by Flopy version 3.9.2 on 04/20/2025 at 12:25:36.
BEGIN options
END options

BEGIN timing
  TDIS6  flow.tdis
END timing

BEGIN models
  gwf6  flow.nam  flow
END models

BEGIN exchanges
END exchanges

BEGIN solutiongroup  1
  ims6  flow.ims  flow
END solutiongroup  1
```

* Observa que ahora este archivo contiene la información de las tres componentes necesarias para iniciar una simulación: Timing, Models y SolutionGroup. En este ejemplo no se agregan intercambios, pues se trata de un solo modelo.

Más información acerca de los archivos de entrada y salida de MODFLOW 6 se puede encontrar en [5].

</div>

En este ejemplo se ha construido el esquema requerido para iniciar una simulación de flujo con GWF de MODFLOW 6 usando las herramientas de flopy. Para que la simulación tenga más sentido, se requiere de agregar paquetes al modelo GWF.

# Referencias

[1] Hughes, J.D., Langevin, C.D., and Banta, E.R., 2017, Documentation for the MODFLOW 6 framework: U.S. Geological Survey Techniques and Methods, book 6, chap. A57, 40 p. https://doi.org/10.3133/tm6A57.

[2] Langevin, C. D., Hughes, J. D., Provost, A. M., Russcher, M. J., & Panday, S. (2023). MODFLOW as a configurable Multi‐Model Hydrologic Simulator. Ground Water. https://doi.org/10.1111/gwat.13351.

[3] Hughes, J.D., and White, J.T., 2013, Use of general purpose graphics processing units with MODFLOW: Groundwater, v. 51, no. 6, p. 833–846, accessed June 27, 2017. https://doi.org/10.1111/gwat.12004.

[4] Langevin, C.D., Hughes, J.D., Provost, A.M., Banta, E.R., Niswonger, R.G., and Panday, Sorab, 2017, Documentation for the MODFLOW 6 Groundwater Flow (GWF) Model: U.S. Geological Survey Techniques and Methods, book 6, chap. A55, 197 p., accessed August 4, 2017. https://doi.org/10.3133/tm6A55.

[5] MODFLOW 6 – Description of Input and Output. Version mf6.4.4—February 13, 2024. U.S. Department of the Interior. U.S. Geological Survey.. Archivo: mf6io.pdf de la documentación de MODFLOW 6 que se puede obtener de https://github.com/MODFLOW-ORG/modflow6/releases.